In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
sys.path.insert(0, str(Path.cwd().parent / 'python'))
from analysis_common import analytical_src, segment_metrics
from config import AGGREGATE_DIR, AUDIT_DIR, CLEAN_PARQUET, DOCS_DIR, STATS_DIR, VISUALS_DIR, connect
pd.set_option('display.max_columns', 40, 'display.width', 180, 'display.float_format', '{:,.4f}'.format)
con = connect(); src = analytical_src()
kpi = json.loads((AGGREGATE_DIR / 'portfolio_kpis.json').read_text())
fin = json.loads((AGGREGATE_DIR / 'financial_totals.json').read_text())
show = lambda name: display(Image(filename=str(VISUALS_DIR / f'{name}.png')))
csv = lambda name: pd.read_csv(AGGREGATE_DIR / f'{name}.csv')

# 1. Dataset Understanding
Profile of the raw LendingClub file before any change. Produced by `python/data_inspection.py`.

In [2]:
facts = json.loads((AUDIT_DIR / 'audit_facts.json').read_text()); facts

{'csv_bytes': 1675133810,
 'csv_records_excl_header': 2260701,
 'columns': 151,
 'distinct_id': 2260701,
 'null_id': 0,
 'non_numeric_id_or_null_amount_rows': 33,
 'duplicated_ids': 0,
 'distinct_full_row_hashes': 2260701}

2,260,701 records include 33 non-loan rows. No duplicated ids or exact duplicates exist.

In [3]:
pd.read_csv(AUDIT_DIR / 'non_loan_rows.csv').head(6)

,id,member_id,loan_amnt,loan_status
0,Total amount funded in policy code 1: 1791201400,NaN,NaN,NaN
1,Total amount funded in policy code 2: 651669342,NaN,NaN,NaN
2,Total amount funded in policy code 1: 2050909275,NaN,NaN,NaN
3,Total amount funded in policy code 2: 820109297,NaN,NaN,NaN
4,Total amount funded in policy code 1: 2700702175,NaN,NaN,NaN
5,Total amount funded in policy code 2: 81866225,NaN,NaN,NaN


In [4]:
profile = pd.read_csv(AUDIT_DIR / 'column_profile.csv')
profile.sort_values('missing_pct', ascending=False)[['column_name','missing_pct','approx_distinct']].head(15)

,column_name,missing_pct,approx_distinct
1,member_id,100.0000,0
140,orig_projected_additional_accrued_interest,99.6173,6660
135,hardship_end_date,99.5171,27
134,hardship_start_date,99.5171,28
129,hardship_type,99.5171,1
130,hardship_reason,99.5171,9
131,hardship_status,99.5171,1
132,deferral_term,99.5171,1
142,hardship_last_payment_amount,99.5171,6895
141,hardship_payoff_balance_amount,99.5171,9326


In [5]:
freq = pd.read_csv(AUDIT_DIR / 'categorical_frequencies.csv')
freq[freq.column_name == 'loan_status']

,column_name,value,records
71,loan_status,Fully Paid,1076751
72,loan_status,Current,878317
73,loan_status,Charged Off,268559
74,loan_status,Late (31-120 days),21467
75,loan_status,In Grace Period,8436
76,loan_status,Late (16-30 days),4349
77,loan_status,Does not meet the credit policy. Status:Fully ...,1988
78,loan_status,Does not meet the credit policy. Status:Charge...,761
79,loan_status,Default,40
80,loan_status,NaN,33


Only Fully Paid and Charged Off are terminal outcomes in the primary default rate. Current and Late loans are never counted as defaults.

In [6]:
pd.read_csv(DOCS_DIR / 'data_dictionary.csv').groupby('category').column_name.count().sort_values(ascending=False)

category
Credit history                  81
Hardship and settlement         22
Loan characteristics            11
Loan performance                 9
Interest and repayment           8
Income and financial profile     7
Loan identification              5
Employment                       2
Geographic                       2
Recovery                         2
Borrower demographics            1
Origination dates                1
Name: column_name, dtype: int64